In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os
import gc
import subprocess
import os
import time
from openai import OpenAI
import time
import pandas as pd
from tqdm import tqdm


In [2]:
def api_ai_answer(client, prompt):
    start_time = time.perf_counter()
    response = client.chat.completions.create(
        model="LORA_THE_BEST",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1600,
        temperature=0.0
    )
    
    end_time = time.perf_counter()
    
    # Извлекаем текст ответа
    answer = response.choices[0].message.content
    response_time = end_time - start_time
    completion_tokens = response.usage.completion_tokens
    tokens_per_second = completion_tokens / response_time if response_time > 0 else 0
    return [answer, response_time, completion_tokens,tokens_per_second]

In [3]:
def test_model_adapter(base_model_name, adapter_path,
              command = [
    "vllm", "serve",
    "LORA_THE_BEST",
    "--host", "0.0.0.0",
    "--port", "8001",
    "--api-key", "7b956701760dccad68ef20684d635bd3ee82e2e8fa6287f048b572169f178d51",
    "--served-model-name", "LORA_THE_BEST",
    "--max-model-len", "8192",
    "--max-num-seqs", "4"
]):

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token

    model = PeftModel.from_pretrained(base_model, adapter_path)
    merged_model = model.merge_and_unload()
    output_dir = "LORA_THE_BEST"
    merged_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    del merged_model
    del model
    del base_model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    #print(torch.cuda.memory_summary())

    log_file = open(f"vllm_server_{adapter_path}.log", "w")

    process = subprocess.Popen(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,   # объединяем stderr с stdout
        text=True,                  # работаем со строками
        start_new_session=True      # отделяем процесс, чтобы он не завершился при выходе скрипта
    )
    #time.sleep(40)
    
    # Ждём строку "Application startup complete" в логе
    ready = False
    while not ready:
        time.sleep(1)
        with open(f"vllm_server_{adapter_path}.log", "r") as f:
            if "Application startup complete" in f.read():
                ready = True
                break
    client = OpenAI(
        base_url="http://localhost:8001/v1",
        api_key="7b956701760dccad68ef20684d635bd3ee82e2e8fa6287f048b572169f178d51"
    )

    #start_time = time.perf_counter()
    #response = client.chat.completions.create(
    #    model="LORA_THE_BEST",
    #    messages=[{"role": "user", "content": "Мой кариотип 46,XX,t(5;10)(q12;p22). Что это значит?"}],
    #    max_tokens=1600,
    #    temperature=0.7
    #)
    #end_time = time.perf_counter()
    #answer = response.choices[0].message.content
    #print("Ответ модели:")
    #print(answer)
    #print(f"\nВремя выполнения: {end_time - start_time:.3f} секунд")
    
    df = pd.read_excel("Q_F_T.xlsx")

    outputs = []
    for prompt in tqdm(df['Promts'], desc = "Обработка запросов"):
        #print(k)
        outputs.append(api_ai_answer(client, prompt))

    answers = [item[0] for item in outputs]
    response_times = [item[1] for item in outputs]
    completion_tokens = [item[2] for item in outputs]
    tokens_per_second = [item[3] for item in outputs]

    df["Answers"] = answers
    df["Response times"] = response_times
    df["Completion tokens"] = completion_tokens
    df["Tokens per second"] = tokens_per_second
    df.to_excel("VLLM_TEST.xlsx", index = False)
    
    process.terminate()
    

In [4]:
apdpter_path_dict = {
    "meta-llama_Llama-3.1-8B-Instruct_lora_epochs_10_batch_3_optim_adamw_torch_fused_r_16_alpha_32_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head" : "meta-llama/Llama-3.1-8B-Instruct",
    
}

for k, v in apdpter_path_dict.items():
    print(k, v)
    test_model_adapter(v, k)

meta-llama_Llama-3.1-8B-Instruct_lora_epochs_10_batch_3_optim_adamw_torch_fused_r_16_alpha_32_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head meta-llama/Llama-3.1-8B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Обработка запросов: 100%|██████████| 105/105 [11:21<00:00,  6.49s/it]
